In [141]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [142]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.append(str(Path().resolve().parent / "src"))

from data import (
    GDPHMMPreprocessedSlicesDataset,
    GDPHMM_DOSE_MAX,
    GDPHMM_OAR,
)

In [143]:
# Load dataset with all structure groups
dataset = GDPHMMPreprocessedSlicesDataset(
    dataset_name='gdphmm-hn-vmat',
    split='test',
    structure_groups=None  # Load all groups
)

# Find the specific volume and slice
volume_name = 'HNSCC-01-0006+A4Ac+MOS_15521'
slice_idx = 110

# Find the index in the dataset
matching_indices = dataset.index[
    (dataset.index['volume'] == volume_name) &
    (dataset.index['slice_idx'] == slice_idx)
].index.tolist()

if not matching_indices:
    raise ValueError(f"Volume {volume_name} slice {slice_idx} not found")

idx = matching_indices[0]
print(f"Found at dataset index: {idx}")

Found at dataset index: 6617


In [144]:
# Load the slice and extract all components
structure, dose, info = dataset[idx]

# Extract individual components using the selector
ct = dataset.selector.get_component(structure, 'ct')[0].numpy()  # (H, W)
body = dataset.selector.get_component(structure, 'body')[0].numpy()  # (H, W)
ptv_doses = dataset.selector.get_component(structure, 'ptv_doses').numpy()  # (3, H, W)
ptv_sdm = dataset.selector.get_component(structure, 'ptv_sdm').numpy()  # (3, H, W)
oar_sdm = dataset.selector.get_component(structure, 'oar_sdm').numpy()  # (33, H, W)
beam_plate = dataset.selector.get_component(structure, 'beam_plate')[0].numpy()  # (H, W)
angle_plate = dataset.selector.get_component(structure, 'angle_plate')[0].numpy()  # (H, W)
gt_dose = dose[0].numpy()  # (H, W)

print(f"CT shape: {ct.shape}, range: [{ct.min():.3f}, {ct.max():.3f}]")
print(f"Body shape: {body.shape}, unique: {np.unique(body)}")
print(f"PTV Doses shape: {ptv_doses.shape}, range: [{ptv_doses.min():.3f}, {ptv_doses.max():.3f}]")
print(f"PTV SDM shape: {ptv_sdm.shape}, range: [{ptv_sdm.min():.3f}, {ptv_sdm.max():.3f}]")
print(f"OAR SDM shape: {oar_sdm.shape}, range: [{oar_sdm.min():.3f}, {oar_sdm.max():.3f}]")
print(f"Beam Plate shape: {beam_plate.shape}, range: [{beam_plate.min():.3f}, {beam_plate.max():.3f}]")
print(f"Angle Plate shape: {angle_plate.shape}, unique: {np.unique(angle_plate)}")
print(f"GT Dose shape: {gt_dose.shape}, range: [{gt_dose.min():.3f}, {gt_dose.max():.3f}]")

CT shape: (176, 160), range: [0.000, 0.828]
Body shape: (176, 160), unique: [0. 1.]
PTV Doses shape: (3, 176, 160), range: [0.000, 0.682]
PTV SDM shape: (3, 176, 160), range: [-1.000, 0.360]
OAR SDM shape: (33, 176, 160), range: [0.000, 0.740]
Beam Plate shape: (176, 160), range: [0.000, 0.981]
Angle Plate shape: (176, 160), unique: [0.]
GT Dose shape: (176, 160), range: [0.000, 0.711]


In [145]:
# Debug: check angle plate values
print(f"Angle Plate - min: {angle_plate.min()}, max: {angle_plate.max()}, unique: {np.unique(angle_plate)}")
print(f"Angle Plate has content: {angle_plate.any()}")

if not angle_plate.any():
    # Find a slice with angle plate data for this volume
    print("\nAngle plate is empty for this slice (only exists ±5 slices from isocenter)")
    print("Searching for slices with angle plate data...")
    volume_slices = dataset.index[dataset.index['volume'] == volume_name]
    for test_idx in volume_slices.index[:]:
        test_structure, _, test_info = dataset[test_idx]
        test_angle = dataset.selector.get_component(test_structure, 'angle_plate')[0].numpy()
        if test_angle.any():
            print(f"  Found angle plate at slice {test_info['index']}")
            break
    else:
        print("  No slices with angle plate found")

Angle Plate - min: 0.0, max: 0.0, unique: [0.]
Angle Plate has content: False

Angle plate is empty for this slice (only exists ±5 slices from isocenter)
Searching for slices with angle plate data...
  Found angle plate at slice 59


In [146]:
# Compute global SDM range for consistent color scale (centered at 0)
all_sdm = np.concatenate([ptv_sdm, oar_sdm], axis=0)
sdm_abs_max = max(abs(all_sdm.min()), abs(all_sdm.max()))
sdm_vmin, sdm_vmax = -sdm_abs_max, sdm_abs_max
print(f"SDM range: [{sdm_vmin:.3f}, {sdm_vmax:.3f}]")

SDM range: [-1.000, 1.000]


In [ ]:
# CT
plt.figure()
plt.imshow(ct, cmap='gray')
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ct.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# Body Mask
plt.figure()
plt.imshow(body, cmap='gray')
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/body_mask.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV Dose High
plt.figure()
plt.imshow(ptv_doses[0] * GDPHMM_DOSE_MAX, cmap='hot', vmin=0, vmax=GDPHMM_DOSE_MAX)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_dose_high.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV Dose Mid
plt.figure()
plt.imshow(ptv_doses[1] * GDPHMM_DOSE_MAX, cmap='hot', vmin=0, vmax=GDPHMM_DOSE_MAX)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_dose_mid.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV Dose Low
plt.figure()
plt.imshow(ptv_doses[2] * GDPHMM_DOSE_MAX, cmap='hot', vmin=0, vmax=GDPHMM_DOSE_MAX)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_dose_low.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV SDM High
plt.figure()
plt.imshow(ptv_sdm[0], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_sdm_high.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV SDM Mid
plt.figure()
plt.imshow(ptv_sdm[1], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_sdm_mid.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV SDM Low
plt.figure()
plt.imshow(ptv_sdm[2], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/ptv_sdm_low.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [169]:
# Prefer larger organs for visualization
preferred_oars = ['Brain', 'Posterior_Neck', 'BrainStem_03', 'Lungs', 'SpinalCord_05', 'Esophagus', 'Trachea', 'Mandible-PTV', 'Larynx-PTV']

oars_with_contour = []
# First try preferred large organs
for oar_name in preferred_oars:
    if oar_name in GDPHMM_OAR:
        i = GDPHMM_OAR.index(oar_name)
        if oar_sdm[i].min() != oar_sdm[i].max():
            oars_with_contour.append((i, oar_name))
        if len(oars_with_contour) >= 3:
            break

# Fall back to any OAR with contour if needed
if len(oars_with_contour) < 3:
    for i, oar_name in enumerate(GDPHMM_OAR):
        if (i, oar_name) not in oars_with_contour and oar_sdm[i].min() != oar_sdm[i].max():
            oars_with_contour.append((i, oar_name))
        if len(oars_with_contour) >= 3:
            break

print(f"Selected OARs: {[name for _, name in oars_with_contour]}")

Selected OARs: ['Brain', 'Posterior_Neck', 'Lungs']


In [ ]:
# OAR SDM 1
oar_idx, oar_name = oars_with_contour[0]
plt.figure()
plt.imshow(oar_sdm[oar_idx], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig(f'../imgs/oar_sdm_{oar_name}.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# OAR SDM 2
oar_idx, oar_name = oars_with_contour[1]
plt.figure()
plt.imshow(oar_sdm[oar_idx], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig(f'../imgs/oar_sdm_{oar_name}.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# OAR SDM 3
oar_idx, oar_name = oars_with_contour[2]
plt.figure()
plt.imshow(oar_sdm[oar_idx], cmap='RdBu_r', vmin=sdm_vmin, vmax=sdm_vmax)
plt.axis('off')
plt.margins(0, 0)
plt.savefig(f'../imgs/oar_sdm_{oar_name}.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# Beam Plate
plt.figure()
plt.imshow(beam_plate, cmap='viridis')
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/beam_plate.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# Angle Plate
plt.figure()
plt.imshow(angle_plate, cmap='gray')
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/angle_plate.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# GT Dose
plt.figure()
plt.imshow(gt_dose * GDPHMM_DOSE_MAX, cmap='jet', vmin=0, vmax=GDPHMM_DOSE_MAX)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/gt_dose.png', bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
# PTV SDM High
plt.figure()
plt.imshow(np.random.randn(*gt_dose.shape), cmap='jet', vmin=-2, vmax=2)
plt.axis('off')
plt.margins(0, 0)
plt.savefig('../imgs/noise.png', bbox_inches='tight', pad_inches=0)
plt.show()